# Character-Level Language Diffusion with Flax NNX on TPU

This notebook trains a character-level language diffusion model using the Sherlock Holmes book dataset.
It is optimized for TPU accelerators using Flax NNX and JAX.

In [ ]:
try:
    import jax
    import flax
    import kagglehub
except ImportError:
    !pip install jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
    !pip install flax optax kagglehub

import os
import time
import glob
import jax
import jax.numpy as jnp
import optax
import kagglehub
from flax import nnx
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
import numpy as np

# Initialize TPU Pod distributed cluster (for multi-host communication)
if jax.process_count() == 1:
    try:
        jax.distributed.initialize()
    except Exception:
        pass

# Configure TPU device Mesh (split batch dimension for data parallelism)
devices = jax.devices()
mesh = Mesh(devices, ('batch',))
data_sharding = NamedSharding(mesh, P('batch'))
replicated_sharding = NamedSharding(mesh, P())

# Kaggle authentication for Colab environment
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
    print("Colab environment: Loaded Kaggle API key from Secrets.")
except ImportError:
    print("Local environment: Using existing Kaggle configuration.")
except userdata.SecretNotFoundError:
    print("Warning: KAGGLE_USERNAME or KAGGLE_KEY is not set in Colab Secrets.")

print(f"JAX devices: {jax.devices()}")

## Hyperparameters

In [ ]:
global_batch_size = 256  # Global batch size distributed across all TPUs
block_size = 1024        # Expanded context length
max_iters = 20000        # Increased number of training iterations
eval_interval = 1000
learning_rate = 3e-4
eval_iters = 200
n_embd = 768             # Scaled up parameters (GPT-2 Small/Medium scale)
n_head = 12
n_layer = 12
head_dim = n_embd // n_head

## Data Loading

In [ ]:
import re

path = kagglehub.dataset_download("talesgomes27/sherleck-books")
text = ""
for file in glob.glob(os.path.join(path, "*.txt")):
    with open(file, "r", encoding="utf-8") as f:
        content = f.read()
        # Remove Project Gutenberg header
        match_start = re.search(r"\*\*\*\s*START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*", content, re.IGNORECASE)
        if match_start:
            content = content[match_start.end():]
        # Remove Project Gutenberg footer
        match_end = re.search(r"\*\*\*\s*END OF THE PROJECT GUTENBERG EBOOK", content, re.IGNORECASE)
        if match_end:
            content = content[:match_end.start()]
        
        text += content.strip() + "\n\n"

chars = sorted(list(set(text)))
chars = ["_"] + chars  # Mask token added
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
mask_token_id = stoi["_"]

def encode(s): return [stoi[ch] for ch in s]
def decode(l): return "".join([itos[int(n)] for n in l])

data = jnp.array(encode(text), dtype=jnp.int32)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split, rng):
    dataset = train_data if split == "train" else val_data
    rng1, rng2, rng3 = jax.random.split(rng, 3)
    ix = jax.random.randint(rng1, (global_batch_size,), 0, len(dataset) - block_size)
    x_orig = jnp.stack([dataset[i : i + block_size] for i in ix])
    
    mask_probs = jax.random.uniform(rng2, (global_batch_size, 1))
    mask = jax.random.uniform(rng3, (global_batch_size, block_size)) < mask_probs
    x = jnp.where(mask, mask_token_id, x_orig)
    
    # Distribute (Shard) the generated batch across multiple TPUs
    x = jax.device_put(x, data_sharding)
    x_orig = jax.device_put(x_orig, data_sharding)
    mask = jax.device_put(mask, data_sharding)
    
    return x, x_orig, mask

## Model Definition (Flax NNX)

In [ ]:
def rms_norm(x):
    return x * jax.lax.rsqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + 1e-6)

def apply_rotary_emb(x, cos, sin):
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    return jnp.concatenate([y1, y2], axis=-1)

class MultiHeadAttention(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.c_q = nnx.Linear(n_embd, n_embd, use_bias=False, rngs=rngs)
        self.c_k = nnx.Linear(n_embd, n_embd, use_bias=False, rngs=rngs)
        self.c_v = nnx.Linear(n_embd, n_embd, use_bias=False, rngs=rngs)
        self.c_proj = nnx.Linear(n_embd, n_embd, use_bias=False, rngs=rngs)

    def __call__(self, x, cos_sin):
        B, T, C = x.shape
        q = self.c_q(x).reshape(B, T, n_head, head_dim)
        k = self.c_k(x).reshape(B, T, n_head, head_dim)
        v = self.c_v(x).reshape(B, T, n_head, head_dim)

        cos, sin = cos_sin
        q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)
        q, k = rms_norm(q), rms_norm(k)

        q, k, v = map(lambda t: t.transpose(0, 2, 1, 3), (q, k, v))
        
        # Bidirectional attention (No causal mask)
        attn_weights = jnp.matmul(q, k.transpose(0, 1, 3, 2)) / jnp.sqrt(head_dim)
        attn_weights = jax.nn.softmax(attn_weights, axis=-1)
        y = jnp.matmul(attn_weights, v)

        y = y.transpose(0, 2, 1, 3).reshape(B, T, C)
        return self.c_proj(y)

class MLP(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.c_fc = nnx.Linear(n_embd, 4 * n_embd, use_bias=False, rngs=rngs)
        self.c_proj = nnx.Linear(4 * n_embd, n_embd, use_bias=False, rngs=rngs)

    def __call__(self, x):
        x = self.c_fc(x)
        x = jax.nn.relu(x)**2
        return self.c_proj(x)

class Block(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.attn = MultiHeadAttention(rngs)
        self.mlp = MLP(rngs)

    def __call__(self, x, cos_sin):
        x = x + self.attn(rms_norm(x), cos_sin)
        x = x + self.mlp(rms_norm(x))
        return x

class DiffusionModel(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        self.token_emb = nnx.Embed(vocab_size, n_embd, rngs=rngs)
        self.blocks = [Block(rngs) for _ in range(n_layer)]
        self.lm_head = nnx.Linear(n_embd, vocab_size, use_bias=False, rngs=rngs)
        
        cos, sin = self._precompute_rotary_embeddings(block_size * 2)
        self.cos = nnx.Variable(cos)
        self.sin = nnx.Variable(sin)

    def _precompute_rotary_embeddings(self, seq_len):
        inv_freq = 1.0 / (10000 ** (jnp.arange(0, head_dim, 2) / head_dim))
        t = jnp.arange(seq_len)
        freqs = jnp.outer(t, inv_freq)
        return jnp.cos(freqs)[None, :, None, :], jnp.sin(freqs)[None, :, None, :]

    def __call__(self, idx, targets=None, mask=None):
        B, T = idx.shape
        x = rms_norm(self.token_emb(idx))
        cos_sin = (self.cos.value[:, :T], self.sin.value[:, :T])
        for block in self.blocks: x = block(x, cos_sin)
        logits = self.lm_head(rms_norm(x))

        if targets is not None:
            loss_all = optax.softmax_cross_entropy_with_integer_labels(logits, targets)
            loss = jnp.sum(loss_all * mask) / (jnp.sum(mask) + 1e-6)
            return logits, loss
        return logits, None

## Training Utilities

In [ ]:
rngs = nnx.Rngs(1337)
model = DiffusionModel(rngs)
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate))

@nnx.jit
def train_step(model, optimizer, x, y, mask):
    def loss_fn(model):
        _, loss = model(x, y, mask)
        return loss
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)
    return loss

def estimate_loss(model, rng):
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(10):
            rng, subkey = jax.random.split(rng)
            x, y, mask = get_batch(split, subkey)
            _, loss = model(x, y, mask)
            losses.append(loss)
        out[split] = jnp.mean(jnp.array(losses))
    return out, rng

## Generation (Parallel Decoding)

In [ ]:
def generate(model, max_new_tokens, prompt_len=16, temp=1.0, confidence_threshold=0.95, top_k=3, rng=None):
    if rng is None: rng = jax.random.PRNGKey(42)
    all_tokens = encode(text[:prompt_len])
    
    while len(all_tokens) - prompt_len < max_new_tokens:
        block_len = min(240, prompt_len + max_new_tokens - len(all_tokens))
        x = jnp.full((1, block_size), mask_token_id, dtype=jnp.int32)
        x = x.at[0, :prompt_len].set(jnp.array(all_tokens[-prompt_len:], dtype=jnp.int32))
        masked = jnp.zeros((1, block_size), dtype=jnp.bool_).at[0, prompt_len : prompt_len + block_len].set(True)
        
        while jnp.any(masked):
            logits, _ = model(x)
            probs = jax.nn.softmax(logits / temp, axis=-1)
            top_k_probs, top_k_indices = jax.lax.top_k(probs, k=top_k)
            confidences = jnp.sum(top_k_probs, axis=-1)
            
            decode_mask = (confidences >= confidence_threshold) & masked
            if not jnp.any(decode_mask):
                flat_idx = jnp.argmax(jnp.where(masked, confidences, -1.0))
                decode_mask = decode_mask.at[jnp.unravel_index(flat_idx, decode_mask.shape)].set(True)
            
            rng, subkey = jax.random.split(rng)
            sampled_k = jax.random.categorical(subkey, jnp.log(top_k_probs / jnp.sum(top_k_probs, axis=-1, keepdims=True) + 1e-10).reshape(-1, top_k))
            sampled_tokens = jnp.take_along_axis(top_k_indices, sampled_k.reshape(1, block_size, 1), axis=-1).squeeze(-1)
            
            x = jnp.where(decode_mask, sampled_tokens, x)
            masked = masked & ~decode_mask
            
        all_tokens.extend(x[0, prompt_len : prompt_len + block_len].tolist())
    return decode(all_tokens)

## Training Loop

In [ ]:
print("Training on TPU...")
start_time = time.time()
rng = jax.random.PRNGKey(1337)

for i in range(max_iters):
    if i % eval_interval == 0 or i == max_iters - 1:
        losses, rng = estimate_loss(model, rng)
        print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, time {time.time() - start_time:.2f}s")
        sample = generate(model, 100, rng=rng)
        print(f"Sample: {sample[:100]}...")

    rng, subkey = jax.random.split(rng)
    xb, yb, mb = get_batch("train", subkey)
    loss = train_step(model, optimizer, xb, yb, mb)

print(f"Finished training in {time.time() - start_time:.2f}s")

## Final Generation

In [ ]:
final_output = generate(model, max_new_tokens=500, temp=0.8)
print(f"\nFinal Output:\n{final_output}")

## 🎥 Real-time visualization of diffusion generation process

The code below shows the process of the model generating text by filling in blanks (`█`) in real-time within the notebook.

In [ ]:
from IPython.display import clear_output, display, HTML
import time
import random

def visualize_sherlock_decoding(model, max_new_tokens=500, prompt_len=16, temp=1.0, confidence_threshold=0.95, top_k=3):
    rng = jax.random.PRNGKey(42)
    all_tokens = encode(text[:prompt_len])
    
    # Character set for Sherlock Holmes cryptographic decoupling theme
    crypto_chars = ['@', '#', '$', '%', '&', '*', '?', '!', '~', '+', '=', 'A', 'X', 'Z', 'Q']
    
    # HTML/CSS based dark mode & neon text output
    def display_crypto_state(tokens, block_mask, block_start):
        clear_output(wait=True)
        
        # Already completed past texts (dark gray)
        completed_text = decode(tokens[:block_start])
        completed_html = f"<span style='color: #666666;'>{completed_text.replace(chr(10), '<br>').replace(' ', '&nbsp;')}</span>"
        
        # Block currently being decoupled
        current_html = ""
        for i, token in enumerate(tokens[block_start:]):
            if i < len(block_mask) and block_mask[i]:
                # The masked part feels like it's quickly rotating with random cryptographic characters (blue neon)
                random_char = random.choice(crypto_chars)
                current_html += f"<span style='color: #4a90e2; font-weight: bold; text-shadow: 0px 0px 3px #4a90e2;'>{random_char}</span>"
            else:
                # Characters confirmed with high confidence (crimson highlight)
                char = decode([token])
                if char == "\n":
                    current_html += "<br>"
                elif char == " ":
                    current_html += "&nbsp;"
                else:
                    current_html += f"<span style='color: #d32f2f; font-weight: bold; text-shadow: 0px 0px 5px rgba(211,47,47,0.7);'>{char}</span>"
                
        html_out = f"""
        <div style='font-family: monospace; padding: 25px; background-color: #121212; color: #d4d4d4; border-radius: 10px; border: 1px solid #333; box-shadow: inset 0 0 10px #000;'>
            <h3 style='color: #e0e0e0; border-bottom: 2px dashed #444; padding-bottom: 10px; margin-top: 0;'>
                <span style='font-size: 1.2em;'>🕵️‍♂️</span> Sherlock Holmes: Cryptographic Decoupling...
            </h3>
            <div style='font-size: 16px; line-height: 1.6; word-wrap: break-word;'>
                {completed_html}{current_html}
            </div>
        </div>
        """
        display(HTML(html_out))
        time.sleep(0.05) # Adjust visualization speed (feel of rotating cryptographic characters)

    while len(all_tokens) - prompt_len < max_new_tokens:
        block_start = len(all_tokens)
        block_len = min(240, prompt_len + max_new_tokens - len(all_tokens))
        
        x = jnp.full((1, block_size), mask_token_id, dtype=jnp.int32)
        x = x.at[0, :prompt_len].set(jnp.array(all_tokens[-prompt_len:], dtype=jnp.int32))
        masked = jnp.zeros((1, block_size), dtype=jnp.bool_).at[0, prompt_len : prompt_len + block_len].set(True)
        
        # Temporary token array (for visualization)
        temp_tokens = list(all_tokens) + [mask_token_id] * block_len

        while jnp.any(masked):
            logits, _ = model(x)
            probs = jax.nn.softmax(logits / temp, axis=-1)
            top_k_probs, top_k_indices = jax.lax.top_k(probs, k=top_k)
            confidences = jnp.sum(top_k_probs, axis=-1)
            
            decode_mask = (confidences >= confidence_threshold) & masked
            if not jnp.any(decode_mask):
                flat_idx = jnp.argmax(jnp.where(masked, confidences, -1.0))
                decode_mask = decode_mask.at[jnp.unravel_index(flat_idx, decode_mask.shape)].set(True)
            
            # Sampling logic
            rng, subkey = jax.random.split(rng)
            top_k_probs_norm = top_k_probs / jnp.sum(top_k_probs, axis=-1, keepdims=True)
            
            # Multinomial sampling using Gumbel trick
            gumbel_noise = -jnp.log(-jnp.log(jax.random.uniform(subkey, top_k_probs_norm.shape) + 1e-10))
            sampled_idx = jnp.argmax(jnp.log(top_k_probs_norm + 1e-10) + gumbel_noise, axis=-1)
            
            sampled_tokens = jnp.take_along_axis(top_k_indices, jnp.expand_dims(sampled_idx, axis=-1), axis=-1).squeeze(-1)
            
            x = jnp.where(decode_mask, sampled_tokens, x)
            masked = masked & ~decode_mask
            
            # Update prediction result of current block
            current_block_tokens = np.array(x[0, prompt_len : prompt_len + block_len])
            current_mask = np.array(masked[0, prompt_len : prompt_len + block_len])
            
            temp_tokens[block_start:] = current_block_tokens.tolist()
            display_crypto_state(temp_tokens, current_mask, block_start)

        all_tokens.extend(x[0, prompt_len : prompt_len + block_len].tolist())

    clear_output(wait=True)
    print("\n✨ Cryptographic decoupling complete! (Final Output):\n")
    print(decode(all_tokens))
    
# Visualize the 500 token generation process
visualize_sherlock_decoding(model, max_new_tokens=500)
